[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobMulla/torchtpu-inference-workshop/blob/main/notebooks/03_vllm_torchtpu.ipynb)

# Demo 3: vLLM on TorchTPU

The live demo served Qwen3-32B over four chips with vLLM. Here the same engine serves `Qwen/Qwen3-0.6B` on one chip: install, start the server, one request, a burst of sixteen (continuous batching), then the prefix cache.

**Runtime:** TPU. About ten minutes the first time (the install and the model download). One engine per runtime: stop `02`'s server first, or restart the runtime.

## 1. Install

vLLM-TorchTPU in a Python 3.12 environment of its own: `pip install vllm-torchtpu`, one package that brings a matching vLLM. The notebook stays the client.

In [ ]:
import importlib.util, os

if importlib.util.find_spec("google.colab"):  # Colab's TPU runtime: one chip, no metadata server
    for k, v in {"TPU_SKIP_MDS_QUERY": "1", "TPU_CHIPS_PER_HOST_BOUNDS": "1,1,1",
                 "TPU_HOST_BOUNDS": "1,1,1", "TPU_WORKER_ID": "0", "TPU_WORKER_HOSTNAMES": "localhost"}.items():
        os.environ.setdefault(k, v)  # inherited by the scripts and servers started below

import importlib.util, os, shutil, subprocess, sys

VENV = os.path.abspath("venv-vllm")  # the server's own Python 3.12 environment
PY = os.path.join(VENV, "bin", "python")

if shutil.which("uv") is None:
    %pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
if not os.path.exists(PY):
    print("making", VENV, "and installing vLLM-TorchTPU into it (a few minutes the first time)")
    subprocess.run([UV, "venv", "-q", "--python", "3.12", VENV], check=True)
    subprocess.run([UV, "pip", "install", "-q", "--python", PY,
                    "--index-strategy", "unsafe-best-match",  # the best version across both indexes
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                    "vllm-torchtpu", "tpu-info"], check=True)
%pip install -q openai requests

print(subprocess.run([PY, "-c", "import importlib.metadata as md, sys\n"
                      "print('python            ', sys.version.split()[0])\n"
                      "for p in ['vllm-torchtpu', 'vllm-torchtpu-core', 'torch', 'torch_tpu', 'libtpu']:\n"
                      "    print(f'{p:18s}', md.version(p))"],
                     capture_output=True, text=True).stdout)

assert os.path.exists(os.path.join(VENV, "bin", "vllm")), (
    "vLLM-TorchTPU is not in the environment (an empty placeholder install?): rm -rf " + VENV + " and run this cell again")

## 2. Start the server

`vllm serve` with no TPU flag: the plugin registers itself. `--tensor-parallel-size 1` on one chip, `--max-num-seqs 32`, and `--enable-prompt-tokens-details` so each response reports its cached prompt tokens. The log is `vllm_server.log`.

In [ ]:
import json
import time

MODEL = "Qwen/Qwen3-0.6B"
PORT = 8000
LOG = os.path.abspath("vllm_server.log")
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

env = dict(os.environ,
           PYTHONUNBUFFERED="1",
           VLLM_WORKER_MULTIPROC_METHOD="spawn",
           TORCH_TPU_INTERNAL_MATERIALIZE_COLLECTIVE_TENSORS="false",
           TORCHINDUCTOR_AUTOGRAD_CACHE="0")
if not env.get("TPU_ACCELERATOR_TYPE"):  # e.g. v6e-1, v5litepod-1: the chip tpu-info sees
    chip, n = json.loads(subprocess.run(
        [PY, "-c", "import json; from tpu_info import device; c, n = device.get_local_chips(); "
                   "print(json.dumps([c.value.name, n]))"], capture_output=True, text=True).stdout)
    env["TPU_ACCELERATOR_TYPE"] = {"v5e": "v5litepod", "7x": "tpu7x"}.get(chip, chip) + f"-{n}"
print("TPU_ACCELERATOR_TYPE", env["TPU_ACCELERATOR_TYPE"])

# Colab's sandbox hides /sys/bus/pci, so torch_tpu's device scan finds no TPU there; a
# sitecustomize on the server's PYTHONPATH registers the device anyway (no-op on a VM).
FIX = os.path.abspath("tpu_sitecustomize")
os.makedirs(FIX, exist_ok=True)
with open(os.path.join(FIX, "sitecustomize.py"), "w") as f:
    f.write('# The server\'s own Python 3.12 environment runs with this on PYTHONPATH. Colab\'s sandbox\n# hides /sys/bus/pci, so torch_tpu\'s device scan finds no TPU and never registers the\n# "tpu" device; this registers it anyway. On a TPU VM it does nothing.\ntry:\n    import torch, torch_tpu\n    from torch_tpu import _loader\n    if not _loader._LOADED:\n        _loader._init_device("tpu")\nexcept Exception:\n    pass\n')
env["PYTHONPATH"] = FIX + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [os.path.join(VENV, "bin", "vllm"), "serve", MODEL,
       "--tensor-parallel-size", "1",
       "--host", "127.0.0.1", "--port", str(PORT),
       "--max-model-len", "8192",
       "--max-num-seqs", "32",
       "--max-num-batched-tokens", "2048",
       "--async-scheduling",
       "--enable-prompt-tokens-details"]
print("vllm " + " ".join(cmd[1:]))
proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                        stdin=subprocess.DEVNULL, start_new_session=True)
print("pid", proc.pid, "| log", LOG)

## 3. Wait until it answers

`/health` returns 200 once the model is loaded and compiled. The first start also downloads the weights.

In [ ]:
import requests

t0 = time.time()
ready, last_print = False, -30
while time.time() - t0 < 1800:
    if proc.poll() is not None:
        break
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            ready = True
            break
    except requests.RequestException:  # refused while starting, or /health blocking on a compile
        pass
    if time.time() - t0 - last_print >= 30:
        last_print = time.time() - t0
        with open(LOG) as f:
            lines = f.read().splitlines()
        print(f"[{last_print:5.0f} s] {lines[-1][:110] if lines else ''}")
    time.sleep(1)

START_S = time.time() - t0
if ready:
    print(f"server ready after {START_S:.0f} s")
else:
    print("server not ready; last 40 log lines:")
    with open(LOG) as f:
        print("".join(f.readlines()[-40:]))

## 4. The chip

`tpu-info`: the server holds the chip, and most of its HBM is the model plus the KV cache the engine reserved.

In [ ]:
tpu_info = os.path.join(VENV, "bin", "tpu-info")


def show_chip():  # the chip table and the HBM table; the rest is multi-slice metrics
    out = subprocess.run([tpu_info], capture_output=True, text=True).stdout
    print(out.split("TensorCore Utilization")[0].rstrip())


show_chip()

## 5. One request

The standard `openai` client against the server's endpoint. It prints the time to the first token and the token counts. The first request after a start can take a few seconds, and so can the first burst: the engine compiles once for each new shape it sees, then reuses the program.


In [ ]:
from openai import OpenAI

llm = OpenAI(base_url=BASE_URL, api_key="none")

t0 = time.monotonic()
first = None
stream = llm.chat.completions.create(
    model=MODEL, max_tokens=128,
    messages=[{"role": "user", "content": "In two sentences, what is a KV cache?"}],
    stream=True, stream_options={"include_usage": True},
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})
for chunk in stream:
    for c in chunk.choices:
        if first is None and c.delta.content:
            first = (time.monotonic() - t0) * 1000
        print(c.delta.content or "", end="")
u = chunk.usage  # the last chunk carries the usage
print(f"\n\nfirst token: {first:.0f} ms | prompt {u.prompt_tokens} tokens, "
      f"completion {u.completion_tokens} tokens, {time.monotonic() - t0:.1f} s total")

## 6. A burst of sixteen

Sixteen different questions within one second, all streamed. The engine re-forms the batch every step: a request joins as soon as it arrives and leaves as soon as it finishes, and one weight read serves everyone in the step.

![Continuous batching: requests join and leave every step](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_15.jpg)

In [ ]:
import asyncio
from openai import AsyncOpenAI

N = 16
allm = AsyncOpenAI(base_url=BASE_URL, api_key="none")

asks = ["Explain {} in detail.", "Define {} in one sentence.",
        "List five facts about {}.", "Explain {} in a short paragraph."]
topics = ["torch.compile", "autograd", "nn.Module", "DataLoader", "mixed precision",
          "LayerNorm", "the KV cache", "attention", "dropout", "Adam", "beam search",
          "top-p sampling", "RoPE", "quantization", "tokenizers", "batch norm"]


async def ask(i):
    await asyncio.sleep(i / N)  # arrivals spread over 1 s
    q = asks[i % 4].format(topics[i % 16])
    t0 = time.monotonic()
    stream = await allm.chat.completions.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": q}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    usage = None
    async for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
    dt = time.monotonic() - t0
    print(f"request {i:2d}: {usage.completion_tokens:4d} tokens in {dt:5.1f} s   ({q})")
    return usage.completion_tokens


t_burst = time.monotonic()
tokens = await asyncio.gather(*(ask(i) for i in range(N)))
wall = time.monotonic() - t_burst
BURST_TOK_S = sum(tokens) / wall
print(f"\n{N} requests, {sum(tokens)} completion tokens in {wall:.1f} s "
      f"= {BURST_TOK_S:.0f} tokens/s on this chip")

## 7. The prefix cache

One long document, two questions. The second time the document is already in the cache: `cached` goes from 0 to nearly the whole prompt and the first token comes back sooner.

![Prefix caching: ask about the same document twice, compute once](https://raw.githubusercontent.com/RobMulla/torchtpu-inference-workshop/main/notebooks/img/slide_13.jpg)

In [ ]:
import inspect
import json.decoder

doc = f"{time.time()}\n" + inspect.getsource(json.decoder)
print(f"prompt: {len(doc):,} characters\n")

for q in ["What does JSONDecodeError record?", "What does raw_decode return?"]:
    print(">", q)
    t0, first = time.monotonic(), None
    stream = llm.chat.completions.create(
        model=MODEL, max_tokens=64,
        messages=[{"role": "user", "content": f"{doc}\n\nIn one sentence: {q}"}],
        stream=True, stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    for chunk in stream:
        for c in chunk.choices:
            if first is None and c.delta.content:
                first = (time.monotonic() - t0) * 1000
            print(c.delta.content or "", end="")
    u = chunk.usage
    d = u.prompt_tokens_details  # None when nothing was cached
    cached = d.cached_tokens if d else 0
    print(f"\nfirst token: {first:.0f} ms, cached {cached:,} of {u.prompt_tokens:,} prompt tokens\n")

## Stop the server

Frees the chip.

In [ ]:
import signal


def stop_server(proc, timeout=60):
    if proc.poll() is not None:
        print("already stopped, exit code", proc.returncode)
        return
    os.killpg(proc.pid, signal.SIGINT)
    t0 = time.time()
    while proc.poll() is None and time.time() - t0 < timeout:
        time.sleep(1)
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped after {time.time() - t0:.0f} s, exit code {proc.returncode}")


stop_server(proc)
time.sleep(3)
show_chip()

## Next

- vLLM on TPU: [github.com/vllm-project/vllm-torchtpu](https://github.com/vllm-project/vllm-torchtpu).
- The live demo's files: [`demos/03_vllm/`](https://github.com/RobMulla/torchtpu-inference-workshop/tree/main/demos/03_vllm).